# Project Name: Simple Pairs Trading Strategy Using Cointegration

Eileen QIN Yuxuan, MMath Keble College, Oxford University

配对交易（Pairs Trading）是一种市场中性策略，常用于股票对冲交易。它的核心思想是：找到两只历史走势高度相关的股票，当它们的价格关系出现暂时偏离时，做多被低估的一只，同时做空被高估的一只，等待价差回归，从中获利。

📈 Cointegration testing（协整检验）
- 含义：检验两条时间序列（如两只股票价格）是否存在长期稳定的线性关系。
- 作用：如果两只股票价格走势高度相关但不是完全同步，协整检验能确认它们的价差是否会围绕某个均值波动。
- 在配对交易中的应用：只有通过协整检验的股票对才适合做配对交易，因为它们的价差更可能回归。

⚙️ Parameter optimization（参数优化）
- 含义：在策略中调整关键参数（如开仓阈值、止损比例、回归窗口长度），寻找能最大化收益或风险调整后收益的组合。
- 作用：避免策略过于随意，确保交易规则有统计学和实证支持。
- 例子：设定价差超过 2 个标准差时开仓，超过 0.5 个标准差时平仓，这些阈值就是需要优化的参数。

🔄 Backtesting（回测）
- 含义：用历史数据模拟策略的执行过程，评估其表现。
- 作用：帮助判断策略在过去是否有效，是否能产生稳定收益。
- 注意：回测结果不能保证未来表现，但能揭示策略的潜在问题。

📊 Out-of-sample validation（样本外验证）
- 含义：将数据分为训练集（样本内）和测试集（样本外），在训练集上开发和优化策略，然后在测试集上检验其表现。
- 作用：防止过拟合（策略只在历史数据上有效，但在未来失效）。
- 例子：用 2010–2018 年数据训练策略，用 2019–2021 年数据验证。

💰 Transaction costs（交易成本）
- 含义：执行交易时产生的费用，包括佣金、点差、滑点等。
- 作用：真实交易中成本会显著影响收益，尤其是高频或频繁交易策略。
- 例子：如果每次买卖都要支付 0.1% 的费用，那么策略的理论利润必须足够大才能覆盖这些成本。

👉 总结：
- 协整检验 → 判断股票对是否适合做配对交易。
- 参数优化 → 调整策略规则以提高表现。
- 回测 → 用历史数据检验策略。
- 样本外验证 → 确认策略不是过拟合。
- 交易成本 → 考虑现实中的摩擦，避免纸面利润。



In [12]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
import itertools

# 1. 选择20只科技股（标普500中常见代表性公司）
tickers = [
    "AAPL",
    "MSFT",
    "GOOG",
    "META",
    "NVDA",
    "AMD",
    "INTC",
    "ORCL",
    "IBM",
    "CSCO",
    "ADBE",
    "CRM",
    "TXN",
    "QCOM",
    "AVGO",
    "AMAT",
    "MU",
    "NOW",
    "SNPS",
    "INTU",
]

# 2. 下载过去5年的数据
prices = yf.download(tickers, start="2019-01-01", end="2024-12-31", auto_adjust=False)[
    "Adj Close"
]
prices.to_csv("../data/tech_stocks_prices.csv")

# 3. 协整检验：寻找最具协整性的股票对
pairs = []
for t1, t2 in itertools.combinations(tickers, 2):
    score, pvalue, _ = coint(prices[t1].dropna(), prices[t2].dropna())
    if pvalue < 0.05:  # 显著协整
        pairs.append((t1, t2, pvalue))

pairs_sorted = sorted(pairs, key=lambda x: x[2])
best_pair = pairs_sorted[0]
print("最佳协整股票对:", best_pair)

# 4. 构建价差信号
t1, t2, _ = best_pair
spread = prices[t1] - prices[t2]
zscore = (spread - spread.mean()) / spread.std()

# 参数优化：设定开仓阈值和平仓阈值
entry_threshold = 2.0
exit_threshold = 0.5

# 5. 回测均值回归策略
positions = pd.DataFrame(index=spread.index)
positions[t1] = 0
positions[t2] = 0

# 建立交易信号
long_signal = zscore < -entry_threshold
short_signal = zscore > entry_threshold
exit_signal = abs(zscore) < exit_threshold

positions[t1] = np.where(long_signal, 1, np.where(short_signal, -1, 0))
positions[t2] = -positions[t1]

# 平仓条件
positions[t1] = np.where(exit_signal, 0, positions[t1])
positions[t2] = np.where(exit_signal, 0, positions[t2])

# 6. 计算收益（考虑交易成本）
returns = prices.pct_change().fillna(0)
strategy_ret = (positions.shift().fillna(0) * returns).sum(axis=1)

# 假设每次换仓成本为0.1%
transaction_cost = 0.001
trades = (positions.diff().abs().sum(axis=1) > 0).astype(int)
strategy_ret -= trades * transaction_cost

cum_ret = (1 + strategy_ret).cumprod()
print("策略最终收益:", cum_ret.iloc[-1])

# 7. 样本外验证：前4年训练，最后1年测试
train = strategy_ret[: int(len(strategy_ret) * 0.8)]
test = strategy_ret[int(len(strategy_ret) * 0.8) :]

print("训练期收益:", (1 + train).prod())
print("测试期收益:", (1 + test).prod())

[*********************100%***********************]  20 of 20 completed


最佳协整股票对: ('IBM', 'AVGO', np.float64(0.000315675197370839))
策略最终收益: 1.090154228890049
训练期收益: 1.0
测试期收益: 1.090154228890049


In [4]:
print(find_pairs(data))

[]


## Things Students Should Look At: 

1. 高相关性不等于协整：相关性只衡量波动方向一致，而协整要求差价序列是平稳的。相关性高但不是协整的配对，价差会随时间发散。

2. 样本外表现：配对可能在历史数据中表现良好（过拟合），但在样本外失去协整性，需检查 p-value 在样本外是否仍有效。

3. 阈值敏感性：如果策略仅在特定阈值下盈利，说明策略脆弱。稳健的策略应在阈值微调（如 1.8-2.2）范围内表现平稳。

4. 交易成本：这是实盘的杀手。频繁进出场产生的佣金和点差（slippage）常会将统计套利的微小利润吞噬。

5. 集中度风险：检查是否大部分利润来自某一次极端的回归。如果是，则策略缺乏统计学上的普适性。

6. 现实约束：考虑卖空限制（short selling）、保证金要求（margin calls）和流动性不足导致的滑点。在考虑这些因素后，很多理论上的获利机会将消失。